In [ ]:
!pip -q install -U spacy
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.7/32.7 MB 37.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 93.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


**Part 1 Self-Check Assignment**

<!-- ### Part 1: Self-Check Assignment -->

1. **R** — The course code follows a predictable character pattern.

2. **L** — The dataset names are known exact strings, so literal or phrase matching is sufficient.

3. **R** — Dates follow a predictable numerical and separator-based character pattern.

4. **R** — The expression follows a predictable lexical and numerical structure.

5. **N** — Determining disappointment requires contextual understanding of the review's meaning.

6. **T** — The task depends on a sequence of tokens with an optional adjective.

7. **N** — Determining contradiction requires contextual understanding of the meanings of the two results.

8. **R** — URLs follow a predictable character structure that can be captured using regex.

**Part 2: Regular Expression Foundations**

In [ ]:
import re
from pprint import pprint
import spacy
from spacy.matcher import Matcher, PhraseMatcher
nlp = spacy.load("en_core_web_sm")

In [ ]:
import re

text = """
BERT-base achieved 91.3 F1 on CoNLL-2003.
RoBERTa-large reached 94.8 accuracy on SST-2.
Contact nlp-course@university.edu before 18/08/2026.
"""

match = re.search(r"BERT", text)

if match:
    print(match.group())

BERT


In [ ]:
numbers = re.findall(r"\d+(?:\.\d+)?", text)
print(numbers)

['91.3', '1', '2003', '94.8', '2', '18', '08', '2026']


In [ ]:
model_pattern = re.compile(
r"\b(?:BERT|RoBERTa|T5|DistilBERT)"
r"(?:-(?:base|large|small))?\b",
flags=re.IGNORECASE,
)
for match in model_pattern.finditer(text):
    print(match.group(), match.span())

BERT-base (1, 10)
RoBERTa-large (43, 56)


In [ ]:
re.findall(r"\d+", text)

['91', '3', '1', '2003', '94', '8', '2', '18', '08', '2026']

**Part 2 Self-Check Assignment**

In [ ]:
import re

notices = [
    "NLP-501 begins on 12/08/2026.",
    "Submit Assignment-2 by 18-08-2026.",
    "Contact neural.nlp@dau.ac.in for access.",
    "The room is Lab-204, not Lab204.",
    "NLP501 is not a valid course-code format.",
    "The older NLP-50 code is also invalid."
]

# Task A: Course code
course_code_pattern = re.compile(r"\b[A-Z]{2,4}-\d{3}\b")

for notice in notices:
    matches = course_code_pattern.findall(notice)
    if matches:
        print(matches)

['NLP-501']


In [ ]:
course_code_pattern = re.compile(r"\b(?:NLP|CS|LANG)-\d{3}\b")

for notice in notices:
    matches = course_code_pattern.findall(notice)
    if matches:
        print(matches)

['NLP-501']


In [ ]:
import re

date_pattern = re.compile(r"\b\d{2}[-/]\d{2}[-/]\d{4}\b")

for notice in notices:
    matches = date_pattern.findall(notice)
    if matches:
        print(matches)

['12/08/2026']
['18-08-2026']


In [ ]:
email_pattern = re.compile(r"\b[\w.-]+@[\w.-]+\.\w+\b")

for notice in notices:
    matches = email_pattern.findall(notice)
    if matches:
        print(matches)

['neural.nlp@dau.ac.in']


In [ ]:
import re

# Task A: Course code
course_code_pattern = re.compile(r"\b[A-Z]{2,4}-\d{3}\b")

# Task B: Date
date_pattern = re.compile(r"\b\d{2}[-/]\d{2}[-/]\d{4}\b")

# Task C: Email
email_pattern = re.compile(r"\b[\w.-]+@[\w.-]+\.\w+\b")


# Required output
for notice in notices:
    print({
        "text": notice,
        "course_codes": course_code_pattern.findall(notice),
        "dates": date_pattern.findall(notice),
        "emails": email_pattern.findall(notice),
    })


# ---------------- SELF-CHECK ----------------

# Positive tests
assert course_code_pattern.fullmatch("NLP-501")
assert date_pattern.fullmatch("12/08/2026")
assert email_pattern.fullmatch("neural.nlp@dau.ac.in")

# Negative tests
assert not course_code_pattern.fullmatch("NLP501")
assert not date_pattern.fullmatch("12.08.2026")
assert not email_pattern.fullmatch("neural.nlp@dau")

print("All self-check tests passed!")

{'text': 'NLP-501 begins on 12/08/2026.', 'course_codes': ['NLP-501'], 'dates': ['12/08/2026'], 'emails': []}
{'text': 'Submit Assignment-2 by 18-08-2026.', 'course_codes': [], 'dates': ['18-08-2026'], 'emails': []}
{'text': 'Contact neural.nlp@dau.ac.in for access.', 'course_codes': [], 'dates': [], 'emails': ['neural.nlp@dau.ac.in']}
{'text': 'The room is Lab-204, not Lab204.', 'course_codes': [], 'dates': [], 'emails': []}
{'text': 'NLP501 is not a valid course-code format.', 'course_codes': [], 'dates': [], 'emails': []}
{'text': 'The older NLP-50 code is also invalid.', 'course_codes': [], 'dates': [], 'emails': []}
All self-check tests passed!


 **Part 3: Structured Extraction and Text
Normalization**

In [ ]:
import re

examples = [
    "BERT-base achieved 91.3 F1 on CoNLL-2003.",
    "The dataset contains 12,000 sentences.",
    "T5 obtained 27.6 BLEU on WMT14.",
    "Training used 10 epochs and batch size 32.",
    "The system reached 72% accuracy.",
    "The paper was published in 2024.",
    "BART reported ROUGE-L 41.2.",
]

# Score comes before metric
score_first_pattern = re.compile(
    r"""
    (?P<score>\d{1,3}(?:\.\d+)?)
    \s*
    (?P<percent>%?)
    \s*
    (?P<metric>F1|BLEU|accuracy|ROUGE-L)
    \b
    """,
    re.IGNORECASE | re.VERBOSE
)

# Metric comes before score
metric_first_pattern = re.compile(
    r"""
    (?P<metric>F1|BLEU|accuracy|ROUGE-L)
    \s*
    (?P<score>\d{1,3}(?:\.\d+)?)
    \s*
    (?P<percent>%?)
    """,
    re.IGNORECASE | re.VERBOSE
)


def extract_result(text: str):
    """
    Return a dictionary containing score and metric.
    Return None when no result expression is present.
    """

    # Try score-first pattern
    match = score_first_pattern.search(text)

    # If not found, try metric-first pattern
    if not match:
        match = metric_first_pattern.search(text)

    if not match:
        return None

    result = match.groupdict()

    return {
        "score": float(result["score"]),
        "metric": result["metric"],
        "has_percent_sign": result["percent"] == "%"
    }


# Test all given examples
for example in examples:
    print(example)
    print(extract_result(example))
    print()

BERT-base achieved 91.3 F1 on CoNLL-2003.
{'score': 91.3, 'metric': 'F1', 'has_percent_sign': False}

The dataset contains 12,000 sentences.
None

T5 obtained 27.6 BLEU on WMT14.
{'score': 27.6, 'metric': 'BLEU', 'has_percent_sign': False}

Training used 10 epochs and batch size 32.
None

The system reached 72% accuracy.
{'score': 72.0, 'metric': 'accuracy', 'has_percent_sign': True}

The paper was published in 2024.
None

BART reported ROUGE-L 41.2.
{'score': 41.2, 'metric': 'ROUGE-L', 'has_percent_sign': False}



**Part 4: spaCy Text Representation and Tokenization**

In [ ]:
# ============================================
# Part 4: spaCy Text Representation
# Self-Check Assignment
# ============================================

# Install spaCy and English model in Colab
!pip install -q spacy
!python -m spacy download en_core_web_sm

import spacy

# Load English spaCy pipeline
nlp = spacy.load("en_core_web_sm")

# Given examples
samples = [
    "BERT-base achieved 91.3% accuracy.",
    "The U.K.-based team used GPT-2.",
    "The model doesn't tokenize naïvely.",
    "Results are available at nlp.example.org/results.",
    "RoBERTa-large outperformed BERT-base—slightly.",
]

# ============================================
# 1. Token Inspection Output
# ============================================

for text in samples:
    print("\n" + "=" * 80)
    print("TEXT:", text)
    print("=" * 80)

    doc = nlp(text)

    for token in doc:
        print({
            "token": token.text,
            "index": token.i,
            "character_offset": token.idx,
            "lowercase": token.lower_,
            "lemma": token.lemma_,
            "pos": token.pos_,
            "is_punctuation": token.is_punct,
            "looks_like_number": token.like_num,
            "looks_like_url": token.like_url,
        })


# ============================================
# 2. Answers to the Questions
# ============================================

print("\n\n" + "=" * 80)
print("ANSWERS TO THE EIGHT QUESTIONS")
print("=" * 80)

print("""
1. How is BERT-base tokenized?
   BERT-base is tokenized as:
   BERT | - | base

2. How is GPT-2 tokenized?
   GPT-2 is tokenized as:
   GPT | - | 2

3. Is 91.3 recognized as number-like?
   Yes. The token 91.3 has token.like_num = True.

4. How is doesn't segmented?
   It is segmented as:
   does | n't

5. Which token or tokens are identified as URL-like?
   The token:
   nlp.example.org/results
   is identified as URL-like.

6. How does the em dash affect tokenization?
   The em dash (—) is treated as a separate punctuation token.
   For example:
   BERT | - | base | — | slightly

7. Why could a token-level pattern fail even when the visible
   text appears correct?
   A token-level pattern can fail because the visible text and
   spaCy's token boundaries may be different. For example,
   BERT-base appears as one visible expression but spaCy
   tokenizes it as BERT, -, and base.

8. What is the difference between token.i and token.idx?
   token.i = position/index of the token inside the spaCy Doc.
   token.idx = character offset where the token starts in the
   original text.
""")


# ============================================
# 3. One Additional Unexpected Tokenization
# ============================================

print("\n" + "=" * 80)
print("ADDITIONAL UNEXPECTED TOKENIZATION")
print("=" * 80)

extra = "state-of-the-art"
doc = nlp(extra)

print("Text:", extra)
print("Tokens:", [token.text for token in doc])

print("""
Observation:
The visually connected expression 'state-of-the-art' is split
into separate tokens around the hyphens.
""")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 91.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

TEXT: BERT-base achieved 91.3% accuracy.
{'token': 'BERT', 'index': 0, 'character_offset': 0, 'lowercase': 'bert', 'lemma': 'BERT', 'pos': 'PROPN', 'is_punctuation': False, 'looks_like_number': False, 'looks_like_url': False}
{'token': '-', 'index': 1, 'character_offset': 4, 'lowercase': '-', 'lemma': '-', 'pos': 'PUNCT', 'is_punctuation': True, 'looks_like_number': False, 'looks_like_url': False}
{'token': 'base', 'index': 2, 'character_offset': 5, 'lowercase': 'base', 'lemma': 'base', 'pos': 'PROPN', 'is_punctuation': False, 'looks_like_number': False, 'looks_like_url': Fal

**Part 5: PhraseMatcher and Matcher**

In [ ]:
# ============================================================
# PART 5 SELF-CHECK ASSIGNMENT
# PhraseMatcher and Matcher
# ============================================================

!pip install -q spacy
!python -m spacy download en_core_web_sm

import spacy
from spacy.matcher import PhraseMatcher, Matcher

nlp = spacy.load("en_core_web_sm")


# ============================================================
# TASK A: Known Terminology using PhraseMatcher
# ============================================================

models = [
    "BERT",
    "BERT-base",
    "RoBERTa-large",
    "T5-small",
    "DistilBERT",
    "BiLSTM-CRF"
]

datasets = [
    "CoNLL-2003",
    "SST-2",
    "SQuAD 2.0",
    "AG News",
    "WMT14"
]

phrase_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

model_patterns = [nlp.make_doc(x) for x in models]
dataset_patterns = [nlp.make_doc(x) for x in datasets]

phrase_matcher.add("MODEL", model_patterns)
phrase_matcher.add("DATASET", dataset_patterns)


# Test Task A
task_a_text = """
BERT-base was evaluated on SST-2.
RoBERTa-large was compared with T5-small.
The system also used DistilBERT and BiLSTM-CRF.
Results were compared with CoNLL-2003, SQuAD 2.0, AG News, and WMT14.
"""

doc = nlp(task_a_text)

print("=" * 70)
print("TASK A: PHRASEMATCHER")
print("=" * 70)

for match_id, start, end in phrase_matcher(doc):
    label = nlp.vocab.strings[match_id]
    span = doc[start:end]
    print((label, span.text))


# ============================================================
# TASK B: Flexible Model Pattern using Matcher
# ============================================================

matcher = Matcher(nlp.vocab)

model_pattern = [
    {
        "LOWER": {
            "IN": ["bert", "roberta", "t5"]
        }
    },
    {
        "TEXT": "-",
        "OP": "?"
    },
    {
        "LOWER": {
            "IN": ["small", "base", "large"]
        },
        "OP": "?"
    }
]

matcher.add("MODEL", [model_pattern])


# Test model pattern
model_examples = [
    "BERT",
    "BERT-base",
    "BERT-large",
    "RoBERTa-base",
    "RoBERTa-large",
    "T5-small",
    "T5-base",
    "T5-large"
]

print("\n" + "=" * 70)
print("TASK B: FLEXIBLE MODEL PATTERN")
print("=" * 70)

for text in model_examples:
    doc = nlp(text)

    print("\nText:", text)
    print("Tokens:", [token.text for token in doc])

    matches = matcher(doc)

    for match_id, start, end in matches:
        print("Match:", doc[start:end].text)


# ============================================================
# TASK C: Reported Result Pattern
# ============================================================

result_matcher = Matcher(nlp.vocab)

result_pattern = [
    # Reporting verb
    {
        "LEMMA": {
            "IN": [
                "achieve",
                "reach",
                "obtain",
                "report",
                "score"
            ]
        }
    },

    # Optional "a"
    {
        "LOWER": "a",
        "OP": "?"
    },

    # Optional "score"
    {
        "LOWER": "score",
        "OP": "?"
    },

    # Optional "of"
    {
        "LOWER": "of",
        "OP": "?"
    },

    # Number
    {
        "LIKE_NUM": True
    },

    # Optional percentage sign
    {
        "TEXT": "%",
        "OP": "?"
    },

    # Metric
    {
        "LOWER": {
            "IN": [
                "f1",
                "accuracy",
                "bleu",
                "rouge-l"
            ]
        }
    }
]

result_matcher.add("REPORTED_RESULT", [result_pattern])


# ============================================================
# TEST CORPUS
# ============================================================

matcher_examples = [
    "BERT-base achieved 91.3 F1.",
    "RoBERTa-large reached 72% accuracy.",
    "T5-small obtained 27.6 BLEU.",
    "BART reported a score of 41.2 ROUGE-L.",
    "The model trained for 20 epochs.",
    "The dataset contains 12,000 sentences.",
    "The work was published in 2024.",
]


print("\n" + "=" * 70)
print("TASK C: REPORTED RESULT")
print("=" * 70)

for text in matcher_examples:

    doc = nlp(text)

    print("\nText:", text)

    # MODEL matches
    model_matches = matcher(doc)

    for match_id, start, end in model_matches:
        label = nlp.vocab.strings[match_id]
        print((label, doc[start:end].text))

    # REPORTED_RESULT matches
    result_matches = result_matcher(doc)

    for match_id, start, end in result_matches:
        label = nlp.vocab.strings[match_id]
        print((label, doc[start:end].text))


# ============================================================
# REQUIRED SELF-CHECK
# ============================================================

print("\n" + "=" * 70)
print("SELF-CHECK")
print("=" * 70)

print("""
Requirements satisfied:

1. At least three token attributes:
   - LOWER
   - LEMMA
   - LIKE_NUM
   - TEXT

2. Optional tokens:
   - "-" in the model pattern
   - "a", "score", "of", "%" in the result pattern

3. At least three reporting verbs:
   - achieve
   - reach
   - obtain
   - report
   - score

4. Positive examples:
   - achieved 91.3 F1
   - reached 72% accuracy
   - obtained 27.6 BLEU
   - reported a score of 41.2 ROUGE-L

5. Negative examples:
   - trained for 20 epochs
   - contains 12,000 sentences
   - published in 2024

The negative examples should NOT produce REPORTED_RESULT matches.
""")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 66.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
TASK A: PHRASEMATCHER
('MODEL', 'BERT')
('MODEL', 'BERT-base')
('DATASET', 'SST-2')
('MODEL', 'RoBERTa-large')
('MODEL', 'T5-small')
('MODEL', 'DistilBERT')
('MODEL', 'BiLSTM-CRF')
('DATASET', 'CoNLL-2003')
('DATASET', 'SQuAD 2.0')
('DATASET', 'AG News')
('DATASET', 'WMT14')

TASK B: FLEXIBLE MODEL PATTERN

Text: BERT
Tokens: ['BERT']
Match: BERT

Text: BERT-base
Tokens: ['BERT', '-', 'base']
Match: BERT
Match: BERT-
Match: BERT-base

Text: BERT-large
Tokens: ['BERT', '-', 'large']
Match: BERT
Match: BERT-
Match: BERT-large

Text: RoBERTa-base
Tokens: ['RoBERTa', '-', 'base']


In [ ]:
# ============================================================
# PART 6: Integrated Rule-Based NLP Challenge
# ============================================================

!pip install -q spacy
!python -m spacy download en_core_web_sm

import spacy
import re
from spacy.matcher import PhraseMatcher, Matcher

nlp = spacy.load("en_core_web_sm")


# ============================================================
# 1. Known Models and Datasets
# ============================================================

models = [
    "BERT",
    "BERT-base",
    "BERT-large",
    "RoBERTa-base",
    "RoBERTa-large",
    "T5-small",
    "T5-base",
    "T5-large",
    "DistilBERT",
    "GPT-2",
]

datasets = [
    "CoNLL-2003",
    "SST-2",
    "SQuAD 2.0",
    "AG News",
    "WMT14",
    "WMT14 En-De",
]


# ============================================================
# 2. PhraseMatcher
# ============================================================

phrase_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

model_patterns = [nlp.make_doc(model) for model in models]
dataset_patterns = [nlp.make_doc(dataset) for dataset in datasets]

phrase_matcher.add("MODEL", model_patterns)
phrase_matcher.add("DATASET", dataset_patterns)


# ============================================================
# 3. Matcher for Reporting Expressions
# ============================================================

report_matcher = Matcher(nlp.vocab)

report_pattern = [
    {
        "LEMMA": {
            "IN": [
                "achieve",
                "reach",
                "obtain",
                "score",
                "report"
            ]
        }
    },

    # Optional "a"
    {
        "LOWER": "a",
        "OP": "?"
    },

    # Optional "score"
    {
        "LOWER": "score",
        "OP": "?"
    },

    # Optional "of"
    {
        "LOWER": "of",
        "OP": "?"
    },

    # Number
    {
        "LIKE_NUM": True
    },

    # Optional %
    {
        "TEXT": "%",
        "OP": "?"
    },

    # Metric
    {
        "LOWER": {
            "IN": [
                "f1",
                "accuracy",
                "bleu",
                "rouge-l"
            ]
        }
    }
]

report_matcher.add("REPORTED_RESULT", [report_pattern])


# ============================================================
# 4. Regex for Numerical Metric Expressions
# ============================================================

metric_regex = re.compile(
    r"\b\d+(?:\.\d+)?\s*%?\s*"
    r"(?:F1|accuracy|BLEU|ROUGE-L)\b",
    re.IGNORECASE
)


# ============================================================
# 5. Main Extraction Function
# ============================================================

def extract_experiment_information(text: str) -> dict:
    """
    Extract model names, dataset names, and reported results.

    Returns:
    {
        "text": text,
        "models": [],
        "datasets": [],
        "results": [
            {
                "score": float,
                "metric": str
            }
        ]
    }
    """

    doc = nlp(text)

    # --------------------------------------------------------
    # PhraseMatcher: Models and datasets
    # --------------------------------------------------------

    found_models = []
    found_datasets = []

    for match_id, start, end in phrase_matcher(doc):

        label = nlp.vocab.strings[match_id]
        span_text = doc[start:end].text

        if label == "MODEL":
            found_models.append(span_text)

        elif label == "DATASET":
            found_datasets.append(span_text)

    # Deduplicate while preserving order
    found_models = list(dict.fromkeys(found_models))
    found_datasets = list(dict.fromkeys(found_datasets))


    # --------------------------------------------------------
    # Matcher: Find reporting expressions
    # --------------------------------------------------------

    reporting_spans = []

    for match_id, start, end in report_matcher(doc):

        span = doc[start:end]

        reporting_spans.append(
            (span.start_char, span.end_char)
        )


    # --------------------------------------------------------
    # Regex: Extract score + metric
    # --------------------------------------------------------

    results = []

    for match in metric_regex.finditer(text):

        # Check whether this metric expression occurs
        # inside a reporting expression.
        metric_start = match.start()
        metric_end = match.end()

        belongs_to_report = False

        for report_start, report_end in reporting_spans:

            if (
                metric_start >= report_start
                and metric_end <= report_end
            ):
                belongs_to_report = True
                break

        if not belongs_to_report:
            continue

        expression = match.group()

        # Extract number
        number_match = re.search(
            r"\d+(?:\.\d+)?",
            expression
        )

        if number_match is None:
            continue

        score = float(number_match.group())

        # Extract metric
        metric_match = re.search(
            r"(F1|accuracy|BLEU|ROUGE-L)",
            expression,
            re.IGNORECASE
        )

        if metric_match is None:
            continue

        metric = metric_match.group()

        # Standardize metric names
        metric_map = {
            "f1": "F1",
            "accuracy": "accuracy",
            "bleu": "BLEU",
            "rouge-l": "ROUGE-L"
        }

        metric = metric_map[metric.lower()]

        results.append({
            "score": score,
            "metric": metric
        })


    # --------------------------------------------------------
    # Deduplicate results
    # --------------------------------------------------------

    unique_results = []

    for result in results:

        if result not in unique_results:
            unique_results.append(result)


    # --------------------------------------------------------
    # Final dictionary
    # --------------------------------------------------------

    return {
        "text": text,
        "models": found_models,
        "datasets": found_datasets,
        "results": unique_results
    }


# ============================================================
# 6. Input Corpus
# ============================================================

mini_corpus = [
    "BERT-base achieved 91.3 F1 on CoNLL-2003.",
    "RoBERTa-large reached 94.8 accuracy on SST-2.",
    "T5-small obtained 27.6 BLEU on WMT14 En-De.",
    "DistilBERT scored 90% accuracy on AG News.",
    "The dataset contains 12,000 sentences and was released in 2003.",
    "GPT-2 generated examples, but no evaluation result was reported.",
]


# ============================================================
# 7. Process Corpus
# ============================================================

print("=" * 80)
print("EXTRACTED EXPERIMENT INFORMATION")
print("=" * 80)

for text in mini_corpus:

    result = extract_experiment_information(text)

    print("\n", result)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 95.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
EXTRACTED EXPERIMENT INFORMATION

 {'text': 'BERT-base achieved 91.3 F1 on CoNLL-2003.', 'models': ['BERT', 'BERT-base'], 'datasets': ['CoNLL-2003'], 'results': [{'score': 91.3, 'metric': 'F1'}]}

 {'text': 'RoBERTa-large reached 94.8 accuracy on SST-2.', 'models': ['RoBERTa-large'], 'datasets': ['SST-2'], 'results': [{'score': 94.8, 'metric': 'accuracy'}]}

 {'text': 'T5-small obtained 27.6 BLEU on WMT14 En-De.', 'models': ['T5-small'], 'datasets': ['WMT14', 'WMT14 En-De'], 'results': [{'score': 27.6, 'metric': 'BLEU'}]}

 {'text': 'DistilBERT scored 90% accuracy on AG News.'

## Part 8: Exit Self-Check

### 1. Give one NLP task where regex is preferable to a neural model.
Regex is preferable for extracting **email addresses, dates, or phone numbers** because these follow predictable character patterns.

### 2. Give one task where PhraseMatcher is preferable to regex.
PhraseMatcher is preferable for identifying a **known list of model or dataset names**, such as `BERT-base`, `RoBERTa-large`, and `CoNLL-2003`.

### 3. Give one task where Matcher is preferable to PhraseMatcher.
Matcher is preferable for detecting **flexible token-level patterns**, such as a reporting expression like `achieved 91.3 F1`, because it can use attributes such as `LEMMA`, `LOWER`, and `LIKE_NUM`.

### 4. Explain why positive examples alone are insufficient for testing a rule.
Positive examples only show that the rule matches cases it is supposed to match. **Negative examples are also required** to ensure that the rule does not incorrectly match unrelated text.

### 5. Identify one failure that motivates contextual neural representations.
A rule-based system may fail when the **same word has different meanings depending on its context**. Contextual neural representations can use surrounding words to determine the intended meaning.

 **Final Takeaway Homework Assessment**
 It's all files are uploaded in my Github Account.
 https://github.com/Tirth-Kansagra/IT594-Deep-Neural-NLP-Applications